# Linear Regression — Kaggle: World Happiness Report

**Dataset:** [World Happiness Report](https://www.kaggle.com/datasets/obaidhere/world-happiness-report)  
**Task:** Predict a country's **Happiness Score** from socio-economic indicators.

**Download first:**
```bash
kaggle datasets download -d obaidhere/world-happiness-report \
    -p ../../data/world_happiness --unzip
```

**Features used:**
- GDP per capita
- Social support
- Healthy life expectancy
- Freedom to make life choices
- Generosity
- Perceptions of corruption

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import sys
sys.path.insert(0, '../..')
from src.utils import set_style, regression_report

set_style()
np.random.seed(42)

## 1. Load & Inspect

In [ ]:
DATA_DIR = Path('../../data/world_happiness')

# The dataset may have different file names per year — glob all CSVs
csv_files = sorted(DATA_DIR.glob('*.csv'))
print('CSV files found:', [f.name for f in csv_files])

# Load and concatenate across years if multiple files exist
dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f)
    dfs.append(df_tmp)

df_raw = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
print(f'\nShape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('Columns:', df_raw.columns.tolist())
print('\nNull counts:\n', df_raw.isnull().sum())
df_raw.describe()

## 2. Clean & Select Features

> Column names vary slightly by year — adjust the mapping below if needed after inspection.

In [ ]:
# Normalise column names (lowercase, replace spaces)
df_raw.columns = (
    df_raw.columns
          .str.lower()
          .str.strip()
          .str.replace(' ', '_')
          .str.replace('-', '_')
)

# Common column candidates for the happiness score target
target_candidates = ['happiness_score', 'score', 'life_ladder', 'happiness.score']
target_col = next((c for c in target_candidates if c in df_raw.columns), None)
print(f'Target column: {target_col}')

# Feature candidates
feature_candidates = [
    'gdp_per_capita', 'economy_(gdp_per_capita)', 'log_gdp_per_capita',
    'social_support', 'family',
    'healthy_life_expectancy', 'health_(life_expectancy)',
    'freedom_to_make_life_choices', 'freedom',
    'generosity',
    'perceptions_of_corruption', 'trust_(government_corruption)',
]
feature_cols = [c for c in feature_candidates if c in df_raw.columns]
print(f'Features found: {feature_cols}')

In [ ]:
# Keep only relevant columns and drop NaNs
keep = [target_col] + feature_cols
df = df_raw[keep].dropna().copy()
print(f'Clean dataset: {df.shape}')
df.head()

## 3. EDA

In [ ]:
# Distribution of happiness score
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[target_col], bins=25, color='#2563EB', edgecolor='white', alpha=0.85)
ax.axvline(df[target_col].mean(), color='#DC2626', lw=2, linestyle='--',
           label=f'Mean = {df[target_col].mean():.2f}')
ax.set_xlabel('Happiness Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Happiness Score')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[keep].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.3)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlated features with target
target_corr = corr[target_col].drop(target_col).sort_values(ascending=False)
print('Correlation with Happiness Score:')
print(target_corr.to_string())

## 4. Build & Train Model

In [ ]:
X = df[feature_cols].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print('Test metrics:')
regression_report(y_test, y_pred)

## 5. Feature Importance (Coefficients)

In [ ]:
coefs = pipe.named_steps['model'].coef_
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': coefs})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#DC2626' if v < 0 else '#2563EB' for v in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (standardised)')
ax.set_title('Feature Importance — Ridge Regression')
plt.tight_layout()
plt.show()

## 6. Predicted vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.7, color='#2563EB', s=40)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Happiness Score')
ax.set_ylabel('Predicted Happiness Score')
ax.set_title('Predicted vs Actual')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Cross-Validation

In [ ]:
cv_r2 = cross_val_score(pipe, X, y, cv=5, scoring='r2')
print(f'5-fold CV R²: {np.round(cv_r2, 4)}')
print(f'Mean R²     : {cv_r2.mean():.4f}  ±  {cv_r2.std():.4f}')

## Recap

Applied linear regression on a real Kaggle dataset:

1. **EDA** — histograms, correlation heatmap to understand feature-target relationships.
2. **Pipeline** — combined `StandardScaler` + `Ridge` to handle varying feature scales.
3. **Coefficients** — revealed which socio-economic factors most drive happiness.
4. **Cross-validation** — gave honest performance estimate without data leakage.

**Next:** [Gradient Descent — Math Intuition →](../04_gradient_descent/01_math_intuition.ipynb)